# Stage 03: Vessel Segmentation

**Status:** Implemented and locally smoke-tested -- pretrained LWNet, inference only. See `SEGMENTATION_ARCHITECTURE.md` Sec 1.2/2 for the full design and its Appendix A.1 for why this reverses an earlier, since-superseded "Baseline U-Net trained within this project" design. This notebook prepares that already-verified implementation for its first real Colab run -- it does not change `vessel_segmentation_model.py` / `vessel_segmentation_inference.py`'s architecture or inference logic, which already passed local smoke testing against real Stage 02 output.

## Objective

Run the pretrained [LWNet](https://github.com/agaldran/lwnet) ("The Little W-Net That Could", Galdrán et al., MIT License) vessel segmentation checkpoint over Stage 02's processed RGB images to produce a single-channel vessel probability map. Unlike every other trainable stage in this pipeline, **this stage has no training workflow of its own** -- it loads a checkpoint the LWNet authors already trained on DRIVE, outside this project entirely.

## Expected Inputs

- Stage 02 processed RGB images (`datasets/<name>/processed/`, Gamma Correction + CLAHE, native resolution -- see `PROJECT_CODE.md`'s Stage 02 Preprocessing Policy). This notebook does **not** run or rerun Stage 02 -- it only reads its already-generated output, and never stages raw datasets.
- The vendored LWNet checkpoint, uploaded to Google Drive at `exported_models/VesselSegmentation/{best_model.pth,config.cfg}` **before** this notebook is run for the first time (one-time manual step -- the checkpoint is not committed to git, mirroring how this repo excludes other trained-model binaries; see `.gitignore` and Section 4 below).

## Expected Outputs

- A `(H, W, 1)` vessel probability map per image, values in `[0, 1]`, matching that image's own `(H, W)` exactly -- consumed directly by Stage 04 (`SEGMENTATION_ARCHITECTURE.md` Sec 7.1's in-memory inference workflow).
- This session's smoke-test probability maps, binary masks, and qualitative visualizations, exported to this run's Drive experiment folder (Section 10).

## Datasets

None. Vessel Segmentation trains on nothing in this project -- DRIVE, CHASE_DB1, and STARE are explicitly **not** staged, downloaded, or referenced anywhere in this notebook (`SEGMENTATION_ARCHITECTURE.md` Sec 1.1). Section 6 discovers already-processed images generically (mirroring Stage 02's own dataset-agnostic discovery); Sections 7-8 then smoke-test two specific, real project datasets (APTOS2019, IDRiD) by name **for reporting clarity only** -- `vessel_segmentation_inference.py` itself contains no dataset-name references anywhere.

## Dependencies

Stage 02 (Image Preprocessing) must have already been run for APTOS2019 and IDRiD -- this notebook reads its output, never regenerates it. First PyTorch-based stage in this repo (`torch`, `torchvision`, `scikit-image`, `scipy` -- added to `requirements.txt`); every other stage is TensorFlow/Keras.

## Workflow

1. Bootstrap
2. Setup
3. Environment verification
4. Model checkpoint staging (Drive -> `/content`, local SSD)
5. Checkpoint/config verification
6. Discovery of existing Stage 02 processed images (generic, not dataset-hardcoded)
7. APTOS2019 smoke inference
8. IDRiD smoke inference
9. Output shape/value verification
10. Export vessel probability maps and binary masks to Drive
11. Final summary

Deliberately **not** Stage 01/02's Setup -> Verification -> Dataset Staging -> Training -> Evaluation -> Export shape -- this stage has no dataset to stage and no training loop (`colab/README.md`'s notebook table).

### 1. Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

### 2. Setup

Mounts Drive, installs `requirements.txt` (now including `torch`/`torchvision`/`scikit-image`/`scipy` for this stage), `cd`s into the repository. Identical call to every other stage notebook -- `setup.py` is not stage-specific.

In [ ]:
import setup

setup_info = setup.setup()
print(setup_info)

### 3. Environment Verification

`require_gpu=False`, unlike Stage 01's training notebook -- this stage's model is tiny (~68k parameters) and runs a handful of forward passes per image; a GPU speeds up large-batch runs but is not required to use this stage correctly. Everything else (Python/TensorFlow version, Drive mount, required packages including the new PyTorch stack) is still checked and still aborts on failure.

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=False,
)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### 4. Model Checkpoint Staging (Drive -> `/content`)

Copies **only** the two vendored LWNet artifacts -- `exported_models/VesselSegmentation/best_model.pth` and `.../config.cfg` on Drive -- to this project's own `models/vessel_segmentation/` on the Colab VM's **local SSD**. `config.VESSEL_SEG_MODEL_DIR` resolves relative to the cloned repository (`/content/diabetic_retinoplasty/models/vessel_segmentation`), which is local disk, not the Drive FUSE mount -- so every later cell's model load reads from local SSD, not Drive, without any extra plumbing.

Not the whole `external/lwnet/` repository, and not routed through `dataset_staging.py` (that module stages *datasets*; two small files need no thread-pool/verification machinery built for tens of thousands of files).

**One-time manual prerequisite:** upload `best_model.pth` (originally `external/lwnet/experiments/wnet_drive/model_checkpoint.pth`) and `config.cfg` to `MyDrive/DiabeticRetinopathy/exported_models/VesselSegmentation/` before running this cell for the first time -- the checkpoint is not committed to git (`.gitignore`), mirroring how this repo already excludes other trained-model binaries.

In [ ]:
import shutil

import config

VESSEL_SEG_DRIVE_DIR = colab_config.DRIVE.exported_model_dir("VesselSegmentation")
CHECKPOINT_FILES = ("best_model.pth", "config.cfg")

os.makedirs(config.VESSEL_SEG_MODEL_DIR, exist_ok=True)
staged_paths = {}
for filename in CHECKPOINT_FILES:
    src = os.path.join(VESSEL_SEG_DRIVE_DIR, filename)
    dst = os.path.join(config.VESSEL_SEG_MODEL_DIR, filename)
    if not os.path.isfile(src):
        raise RuntimeError(
            f"Vessel Segmentation artifact not found on Drive: {src}. Upload the vendored "
            "LWNet checkpoint there first -- see this cell's markdown and "
            "SEGMENTATION_ARCHITECTURE.md Sec 6."
        )
    shutil.copy2(src, dst)
    staged_paths[filename] = dst
    print(f"Staged {src} -> {dst} ({os.path.getsize(dst):,} bytes)")

print(f"\nLocal checkpoint directory (SSD, not Drive): {config.VESSEL_SEG_MODEL_DIR}")

### 5. Checkpoint / Config Verification

Loads the staged checkpoint through this project's own `vessel_segmentation_model.py` / `vessel_segmentation_inference.py` (not ad hoc code -- this logic is unchanged from local smoke testing) and verifies: parameter count matches LWNet's own published ~70k, `model.mode == "eval"` (a plain instance attribute, unrelated to `nn.Module.eval()`), and that a dummy forward pass returns a single prediction tensor rather than the `(x1, x2)` tuple `WNet.forward()` returns when `mode != "eval"`.

In [ ]:
import torch

from vessel_segmentation_inference import DEFAULT_MODEL_PATH, load_vessel_model

vessel_model = load_vessel_model(DEFAULT_MODEL_PATH)
n_params = sum(p.numel() for p in vessel_model.parameters())
model_device = next(vessel_model.parameters()).device
print(f"Loaded checkpoint from {DEFAULT_MODEL_PATH}")
print(f"Parameter count: {n_params:,} (LWNet's own paper reports ~70k for this configuration)")
print(f"Model device: {model_device} (resolved by vessel_segmentation_model.resolve_device() -- "
      "CUDA if this runtime has a GPU, else CPU)")
print(f"model.mode = {vessel_model.mode!r} (must be 'eval')")
print(f"model.training (nn.Module mode) = {vessel_model.training} (must be False)")
assert vessel_model.mode == "eval", "model.mode was not set to 'eval' -- see vessel_segmentation_model.py"
assert vessel_model.training is False, "model.eval() was not called -- see vessel_segmentation_model.py"

# Created on the SAME device the model actually loaded onto (read back from
# the model itself, never hardcoded 'cpu'/'cuda') -- this is exactly the
# check that catches a "model on CUDA, tensor on CPU" mismatch here, before
# it can surface anywhere else in this notebook.
dummy = torch.rand(1, 3, 512, 512, device=model_device)
with torch.no_grad():
    dummy_output = vessel_model(dummy)
assert isinstance(dummy_output, torch.Tensor), (
    f"Expected a single prediction tensor, got {type(dummy_output)} -- "
    "model.mode was not 'eval', so forward() returned the (x1, x2) training tuple."
)
assert dummy_output.shape == (1, 1, 512, 512)
assert dummy_output.device == model_device, (
    f"Output device {dummy_output.device} does not match model device {model_device}."
)
print("Verified: forward() returns a single (1, 1, 512, 512) tensor, not a (x1, x2) tuple.")
print(f"Verified: model and dummy input/output tensors are all on the same device ({model_device}).")

### 6. Discovery of Existing Stage 02 Processed Images

Reads a small sample of already-processed (Stage 02 output) images directly from Drive -- **does not** run Stage 02 again, stage any raw dataset, or touch DRIVE/CHASE_DB1/STARE (not project datasets; `SEGMENTATION_ARCHITECTURE.md` Sec 1.1). Discovery itself stays fully generic, mirroring `stage02_preprocessing.ipynb`'s own `discover_preprocessing_targets()` approach: walk for any `processed/` folder containing images, rather than hardcoding dataset names -- so this works whether Stage 02 has been run for EyeQ, APTOS2019, IDRiD (any/all of its three subtasks), or a future dataset, with no notebook changes.

Sections 7-8 then pick out the two specific datasets this notebook explicitly smoke-tests (APTOS2019, IDRiD) **from the generic results below** -- the discovery function above never mentions either name.

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
MAX_IMAGES_PER_DATASET = 2

def discover_processed_images(datasets_root, max_per_dataset=MAX_IMAGES_PER_DATASET):
    """Generic, dataset-agnostic: finds any 'processed/' folder under
    datasets_root with image files in it. No dataset name is hardcoded --
    mirrors stage02_preprocessing.ipynb's own discovery philosophy."""
    found = {}
    for dirpath, dirnames, filenames in os.walk(datasets_root):
        if os.path.basename(dirpath) != "processed":
            continue
        images = sorted(f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS))
        if not images:
            continue
        label = os.path.relpath(dirpath, datasets_root)
        found[label] = [os.path.join(dirpath, name) for name in images[:max_per_dataset]]
    return found

processed_by_dataset = discover_processed_images(colab_config.DRIVE.datasets_root)

if not processed_by_dataset:
    raise RuntimeError(
        f"No Stage 02 'processed/' output found under {colab_config.DRIVE.datasets_root}. "
        "Run colab/notebooks/stage02_preprocessing.ipynb for APTOS2019 and IDRiD first -- "
        "this stage reads Stage 02's output, it does not generate it."
    )

for label, paths in processed_by_dataset.items():
    print(f"[{label}] {len(paths)} sample image(s): {[os.path.basename(p) for p in paths]}")

total_images = sum(len(paths) for paths in processed_by_dataset.values())
print(f"\nTotal sample images discovered across {len(processed_by_dataset)} dataset folder(s): {total_images}")

### 7. APTOS2019 Smoke Inference

Picks the discovered dataset-folder label containing `"APTOS2019"` out of Section 6's generic results (a notebook-level lookup, not a change to the discovery function or to `vessel_segmentation_inference.py`) and runs Stage 03 on its first sample image.

In [ ]:
import time

from vessel_segmentation_inference import predict_vessel_mask
import numpy as np

aptos_label = next((label for label in processed_by_dataset if "APTOS2019" in label), None)
if aptos_label is None:
    raise RuntimeError(
        "No APTOS2019 entry found among Section 6's discovered 'processed/' folders. "
        "Run Stage 02 for APTOS2019 first."
    )
aptos_image_path = processed_by_dataset[aptos_label][0]
print(f"APTOS2019 sample ({aptos_label}): {aptos_image_path}")

t0 = time.perf_counter()
aptos_result = predict_vessel_mask(aptos_image_path, model=vessel_model)
aptos_wall_seconds = time.perf_counter() - t0

print(f"input_shape={aptos_result['input_shape']}  probability_map.shape={aptos_result['probability_map'].shape}")
print(f"prob min/max/mean = {aptos_result['probability_map'].min():.4f} / "
      f"{aptos_result['probability_map'].max():.4f} / {aptos_result['probability_map'].mean():.4f}")
print(f"tta_used={aptos_result['tta_used']}  threshold_used={aptos_result['threshold_used']} "
      f"({aptos_result['threshold_source']})")
print(f"inference_seconds={aptos_result['inference_seconds']:.2f}  wall_seconds={aptos_wall_seconds:.2f}")

### 8. IDRiD Smoke Inference

Same as Section 7, for the discovered dataset-folder label containing `"IDRiD"` (whichever of its three subtasks -- grading/localization/segmentation -- Stage 02 has processed; the first one discovered is used, printed explicitly below).

In [ ]:
idrid_label = next((label for label in processed_by_dataset if "IDRiD" in label), None)
if idrid_label is None:
    raise RuntimeError(
        "No IDRiD entry found among Section 6's discovered 'processed/' folders. "
        "Run Stage 02 for at least one IDRiD subtask first."
    )
idrid_image_path = processed_by_dataset[idrid_label][0]
print(f"IDRiD sample ({idrid_label}): {idrid_image_path}")

t0 = time.perf_counter()
idrid_result = predict_vessel_mask(idrid_image_path, model=vessel_model)
idrid_wall_seconds = time.perf_counter() - t0

print(f"input_shape={idrid_result['input_shape']}  probability_map.shape={idrid_result['probability_map'].shape}")
print(f"prob min/max/mean = {idrid_result['probability_map'].min():.4f} / "
      f"{idrid_result['probability_map'].max():.4f} / {idrid_result['probability_map'].mean():.4f}")
print(f"tta_used={idrid_result['tta_used']}  threshold_used={idrid_result['threshold_used']} "
      f"({idrid_result['threshold_source']})")
print(f"inference_seconds={idrid_result['inference_seconds']:.2f}  wall_seconds={idrid_wall_seconds:.2f}")

### 9. Output Shape / Value Verification

Checks every property required before trusting either Section 7's or Section 8's output: finite, within `[0, 1]`, matches the input image's own `(H, W)` exactly, and neither entirely zero nor entirely one.

In [ ]:
def verify_output(name, result):
    prob = result["probability_map"]
    binm = result["binary_mask"]
    checks = {
        "finite": bool(np.isfinite(prob).all()),
        "in_range_0_1": bool(prob.min() >= 0.0 and prob.max() <= 1.0),
        "shape_matches_input": prob.shape[:2] == result["input_shape"] == binm.shape[:2],
        "not_all_zero": not bool(np.all(prob == 0)),
        "not_all_one": not bool(np.all(prob == 1)),
    }
    print(f"[{name}]")
    for check_name, passed in checks.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {check_name}")
    assert all(checks.values()), f"{name} output verification failed -- see failing check(s) above."
    return checks

verify_output("APTOS2019", aptos_result)
verify_output("IDRiD", idrid_result)
print("\nAll output shape/value checks PASSED for both datasets.")

### 10. Export Vessel Probability Maps and Binary Masks to Drive

Creates a new, isolated, timestamped experiment folder under `experiments/VesselSegmentation/` on Drive via `experiment_manager.create_experiment()` -- the same mechanism every other stage notebook uses, reused here for a smoke-test run's outputs rather than a training run's checkpoints (this stage has no training run; `metadata.json` records `run_type="smoke_test"` to make that explicit). For each of Section 7/8's images, writes the raw probability map (`.npy`, lossless), the binary mask (`.png`), and the qualitative overlay visualization (`.png`) into that folder's `predictions/` subfolder.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

import experiment_manager

def visualize_prediction(image_path, result, title):
    original = np.array(Image.open(image_path).convert("RGB"))
    prob2d = result["probability_map"][..., 0]
    alpha = prob2d[..., None]
    red = np.zeros_like(original, dtype=np.float32)
    red[..., 0] = 255
    overlay = (original.astype(np.float32) * (1 - 0.6 * alpha) + red * (0.6 * alpha)).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(original); axes[0].set_title("Stage 02 processed"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("Vessel probability overlay"); axes[1].axis("off")
    axes[2].imshow(result["binary_mask"][..., 0], cmap="gray")
    axes[2].set_title(f"Binary mask (t={result['threshold_used']})"); axes[2].axis("off")
    fig.suptitle(title)
    plt.show()
    return fig

experiment = experiment_manager.create_experiment(
    colab_config.DRIVE.experiment_dir("VesselSegmentation"),
    colab_config.REPO_DIR,
    run_type="smoke_test",
    tta=config.VESSEL_SEG_TTA,
    threshold=aptos_result["threshold_used"],
)

export_targets = [
    ("APTOS2019", aptos_image_path, aptos_result),
    ("IDRiD", idrid_image_path, idrid_result),
]

for name, image_path, result in export_targets:
    stem = os.path.splitext(os.path.basename(image_path))[0]
    prob_path = os.path.join(experiment.predictions_dir, f"{name}_{stem}_probability_map.npy")
    mask_path = os.path.join(experiment.predictions_dir, f"{name}_{stem}_binary_mask.png")
    viz_path = os.path.join(experiment.predictions_dir, f"{name}_{stem}_visualization.png")

    np.save(prob_path, result["probability_map"])
    Image.fromarray((result["binary_mask"][..., 0] * 255).astype(np.uint8)).save(mask_path)

    fig = visualize_prediction(image_path, result, f"Stage 03 -- {name} {os.path.basename(image_path)}")
    fig.savefig(viz_path, dpi=120, bbox_inches="tight")
    plt.close(fig)

    print(f"[{name}] exported:")
    print(f"  {prob_path}")
    print(f"  {mask_path}")
    print(f"  {viz_path}")

print(f"\nAll Stage 03 smoke-test outputs exported to: {experiment.predictions_dir}")

### 11. Final Summary

In [ ]:
print("=" * 70)
print("Stage 03: Vessel Segmentation -- Summary")
print("=" * 70)
print(f"Model: pretrained LWNet (wnet, in_c=3, n_classes=1, layers=(8,16,32))")
print(f"Checkpoint: {DEFAULT_MODEL_PATH} ({n_params:,} parameters)")
print(f"Trained by: LWNet's original authors on DRIVE -- NOT trained within this project")
print(f"TTA: {config.VESSEL_SEG_TTA} (config.VESSEL_SEG_TTA / VESSEL_SEG_TTA env var)")
print(f"Binarizing threshold: {aptos_result['threshold_used']} -- LWNet's own DRIVE-tuned reference "
      "default, NOT validated against this project's datasets (SEGMENTATION_ARCHITECTURE.md Sec 2.3)")
print(f"Datasets smoke-tested: APTOS2019 ({aptos_label}), IDRiD ({idrid_label})")
mean_seconds = (aptos_result["inference_seconds"] + idrid_result["inference_seconds"]) / 2
print(f"Mean inference time per image: {mean_seconds:.2f}s "
      "(this runtime's device; a GPU accelerates large-batch runs, not required for correctness)")
print(f"Outputs exported to: {experiment.predictions_dir}")
print()
print("Known caveat: LWNet never sees Gamma/CLAHE in its own training pipeline (only resize + "
      "ToTensor, no mean/std normalization) -- Stage 02's contrast-enhanced output is a documented "
      "distribution shift relative to what this checkpoint was tuned against. Visually inspect "
      "Section 10's exported overlays before trusting this stage's output for any downstream decision.")
print()
print("external/lwnet/ is no longer required for this stage to run -- vessel_segmentation_model.py "
      "and vessel_segmentation_inference.py are self-contained.")